# Modeling summary

Reads the saved test-set metrics for every model (`metrics_Butina.pkl`, `metrics_Family.pkl` - written by `03_Modeling_GC.ipynb` and `04_Modeling_ML.ipynb`), grouped by feature set (`description`: `GC`, `Descriptors`, `Morgan`) and ranked from best to worst `r2_score`.

In [5]:
import pandas as pd
import pickle

SPLITS = ['Butina', 'Family']

COLUMN_ORDER = ['description', 'algorithm', 'MSE', 'RMSE', 'MAE', 'SD_error', 'NRMSE_%', 'WAPE_%', 'r2_score']

metrics = {}
for s in SPLITS:
    df = pd.read_pickle(rf'Data and variables/metrics_{s}.pkl').drop(columns='MAPE_%', errors='ignore')
    df['WAPE_%'] = df['MAE'] * df['NRMSE_%'] / df['RMSE']
    metrics[s] = df[COLUMN_ORDER]
    print(s, metrics[s].shape)

Butina (30, 9)
Family (30, 9)


## Per split, grouped by feature set

In [6]:
def ranked(df):
    """Sort by description (feature set), best r2 first within each group."""
    return df.sort_values(['description', 'r2_score'], ascending=[True, False]).reset_index(drop=True)


for s in SPLITS:
    print(f'===== {s} split =====')
    display(ranked(metrics[s]).round(3))

===== Butina split =====


,description,algorithm,MSE,RMSE,MAE,SD_error,NRMSE_%,WAPE_%,r2_score
0,Descriptors,MLP_PyTorch,11.162359,3.341012,1.960019,3.34072,15.163198,8.895555,0.911582
1,Descriptors,XGBoost,12.13212,3.483119,1.971473,3.477951,15.808154,8.947541,0.903901
2,Descriptors,GradientBoosting,12.422363,3.524537,1.960977,3.516391,15.99613,8.899903,0.901602
3,Descriptors,SVR,13.1236,3.622651,2.012031,3.614743,16.441419,9.131611,0.896047
4,Descriptors,MLP,13.593887,3.686989,2.004373,3.681039,16.733417,9.096855,0.892322
5,Descriptors,RandomForest,14.653414,3.827978,2.17021,3.817407,17.373296,9.849507,0.88393
6,Descriptors,GaussianProcess,19.079359,4.367993,2.442907,4.35414,19.824155,11.087147,0.848871
7,Descriptors,DecisionTree,24.764026,4.976347,3.087593,4.967554,22.585174,14.013058,0.803843
8,Descriptors,Ridge,28.529893,5.341338,3.687888,5.341507,24.241689,16.737499,0.774013
9,Descriptors,LinearRegression,28.750779,5.361975,3.69516,5.362161,24.335351,16.770502,0.772264


===== Family split =====


,description,algorithm,MSE,RMSE,MAE,SD_error,NRMSE_%,WAPE_%,r2_score
0,Descriptors,XGBoost,12.673114,3.559932,2.03272,3.549482,16.272529,9.291609,0.89182
1,Descriptors,RandomForest,13.155799,3.627092,2.159858,3.623848,16.579522,9.872762,0.8877
2,Descriptors,GradientBoosting,14.562817,3.816126,2.176133,3.801668,17.443599,9.947153,0.875689
3,Descriptors,MLP_PyTorch,14.656965,3.828442,2.197889,3.813854,17.499894,10.046603,0.874886
4,Descriptors,MLP,15.323083,3.914471,2.188752,3.912758,17.893136,10.004835,0.8692
5,Descriptors,GaussianProcess,15.982873,3.997859,2.359143,3.995966,18.274303,10.783698,0.863568
6,Descriptors,SVR,16.394766,4.049045,2.144289,4.046999,18.508277,9.801593,0.860052
7,Descriptors,DecisionTree,23.626783,4.860739,3.144635,4.858931,22.218549,14.374199,0.798318
8,Descriptors,Ridge,32.552508,5.705481,3.751229,5.705417,26.079882,17.146953,0.722126
9,Descriptors,LinearRegression,33.195143,5.761523,3.744843,5.761313,26.336052,17.117761,0.716641


## Best model per feature set

The top row of each `description` group above, i.e. the best algorithm for each feature set, side by side for both splits.

In [7]:
for s in SPLITS:
    best = metrics[s].loc[metrics[s].groupby('description')['r2_score'].idxmax()]
    best = best.sort_values('r2_score', ascending=False).reset_index(drop=True)
    print(f'===== {s} split: best per feature set =====')
    display(best.round(3))

===== Butina split: best per feature set =====


,description,algorithm,MSE,RMSE,MAE,SD_error,NRMSE_%,WAPE_%,r2_score
0,Descriptors,MLP_PyTorch,11.162359,3.341012,1.960019,3.34072,15.163198,8.895555,0.911582
1,GC,MLP_PyTorch,13.259545,3.641366,2.123016,3.6317,16.526357,9.635317,0.89497
2,Morgan,MLP,19.384876,4.402826,2.818431,4.381843,19.982247,12.791465,0.846451


===== Family split: best per feature set =====


,description,algorithm,MSE,RMSE,MAE,SD_error,NRMSE_%,WAPE_%,r2_score
0,Descriptors,XGBoost,12.673114,3.559932,2.03272,3.549482,16.272529,9.291609,0.89182
1,GC,MLP_PyTorch,13.129657,3.623487,2.217139,3.574069,16.563041,10.134591,0.887923
2,Morgan,MLP,16.497235,4.061679,2.709585,4.050823,18.566027,12.385575,0.859177


## Combined view (both splits, best to worst r2)

In [8]:
combined = pd.concat(
    [metrics[s].assign(split=s) for s in SPLITS],
    ignore_index=True,
)
combined = combined.sort_values(['description', 'r2_score'], ascending=[True, False]).reset_index(drop=True)
display(combined.round(3))

,description,algorithm,MSE,RMSE,MAE,SD_error,NRMSE_%,WAPE_%,r2_score,split
0,Descriptors,MLP_PyTorch,11.162359,3.341012,1.960019,3.34072,15.163198,8.895555,0.911582,Butina
1,Descriptors,XGBoost,12.13212,3.483119,1.971473,3.477951,15.808154,8.947541,0.903901,Butina
2,Descriptors,GradientBoosting,12.422363,3.524537,1.960977,3.516391,15.99613,8.899903,0.901602,Butina
3,Descriptors,SVR,13.1236,3.622651,2.012031,3.614743,16.441419,9.131611,0.896047,Butina
4,Descriptors,MLP,13.593887,3.686989,2.004373,3.681039,16.733417,9.096855,0.892322,Butina
5,Descriptors,XGBoost,12.673114,3.559932,2.03272,3.549482,16.272529,9.291609,0.89182,Family
6,Descriptors,RandomForest,13.155799,3.627092,2.159858,3.623848,16.579522,9.872762,0.8877,Family
7,Descriptors,RandomForest,14.653414,3.827978,2.17021,3.817407,17.373296,9.849507,0.88393,Butina
8,Descriptors,GradientBoosting,14.562817,3.816126,2.176133,3.801668,17.443599,9.947153,0.875689,Family
9,Descriptors,MLP_PyTorch,14.656965,3.828442,2.197889,3.813854,17.499894,10.046603,0.874886,Family


## Overall ranking (both splits together, ignoring feature set)

In [9]:
display(combined.sort_values('r2_score', ascending=False).reset_index(drop=True).round(3))

,description,algorithm,MSE,RMSE,MAE,SD_error,NRMSE_%,WAPE_%,r2_score,split
0,Descriptors,MLP_PyTorch,11.162359,3.341012,1.960019,3.34072,15.163198,8.895555,0.911582,Butina
1,Descriptors,XGBoost,12.13212,3.483119,1.971473,3.477951,15.808154,8.947541,0.903901,Butina
2,Descriptors,GradientBoosting,12.422363,3.524537,1.960977,3.516391,15.99613,8.899903,0.901602,Butina
3,Descriptors,SVR,13.1236,3.622651,2.012031,3.614743,16.441419,9.131611,0.896047,Butina
4,GC,MLP_PyTorch,13.259545,3.641366,2.123016,3.6317,16.526357,9.635317,0.89497,Butina
5,Descriptors,MLP,13.593887,3.686989,2.004373,3.681039,16.733417,9.096855,0.892322,Butina
6,Descriptors,XGBoost,12.673114,3.559932,2.03272,3.549482,16.272529,9.291609,0.89182,Family
7,GC,MLP_PyTorch,13.129657,3.623487,2.217139,3.574069,16.563041,10.134591,0.887923,Family
8,Descriptors,RandomForest,13.155799,3.627092,2.159858,3.623848,16.579522,9.872762,0.8877,Family
9,GC,MLP,13.384342,3.658462,2.181232,3.647587,16.722912,9.970461,0.885749,Family
